In [1]:
import jax
import jax.numpy as jnp
import numpy as np

# --- Step 1: Copy your RectangularDecompositionND class here ---
# (Including the _get_level_params and window_fn methods)
# Make sure to also copy the `windows.py` functions if they are in a separate file.

class RectangularDecompositionND:
    # ... your class code ...
    pass

# Helper functions from window_functions.py
def cosine(xmin, xmax, x):
    mu, sd = (xmin+xmax)/2, (xmax-xmin)/2
    ws = ((1+jnp.cos(jnp.pi*(x-mu)/sd))/2)**2
    ws = jnp.heaviside(x-xmin,1)*jnp.heaviside(xmax-x,1)*ws
    w = jnp.prod(ws, axis=0, keepdims=True)
    return w

# --- Step 2: Define the exact inputs that cause the failure ---
TMIN, TMAX = 0.0, 2.0
subdomain_width = 1.0

# The 1x9 grid that fails
subdomain_xs_fail = [np.array([0.0]), np.linspace(TMIN, TMAX, 9)]
subdomain_ws_fail = [
    np.array([2.0]),
    np.array([subdomain_width * (TMAX - TMIN) / (len(subdomain_xs_fail[1]) - 1)] * len(subdomain_xs_fail[1])),
]
unnorm_fail = (np.array([0.0]), np.array([1.0]))


# --- Step 3: Define a simple JIT-compiled function to test the logic ---
@jax.jit
def run_test():
    # A. Get the parameters, just like the trainer does
    static_params, _ = RectangularDecompositionND.init_params(subdomain_xs_fail, subdomain_ws_fail, unnorm_fail)

    # B. Isolate the parameters for a single subdomain (e.g., the first one)
    #    We manually construct the `params` dictionary that window_fn expects.
    subdomain_params_for_one_domain = {
        "static": {
            "decomposition": {
                "subdomain": {
                    "params": [p[0] for p in static_params["subdomain"]["params"]]
                }
            }
        }
    }

    # C. Define a single point to test
    x_point = jnp.array([0.0, 1.0])

    # D. Call the window function
    ws = RectangularDecompositionND.window_fn(subdomain_params_for_one_domain, x_point)
    
    return ws

# --- Step 4: Run the test and print the result ---
try:
    result_ws = run_test()
    print("Test completed successfully.")
    print("Window value (ws) for a single subdomain and point:", result_ws)
except Exception as e:
    print("Test failed with an error:")
    print(e)

Test failed with an error:
type object 'RectangularDecompositionND' has no attribute 'init_params'
